In [ ]:

# Task 1: FAQ data + keyword search (fixed, stopword-aware)

faqs = [
    {"question": "How do I reset my password?",
     "keywords": ["password", "reset", "forgot", "login", "credentials"],
     "answer": "Go to Settings > Security > Reset Password and follow the email link."},

    {"question": "How do I update my email address?",
     "keywords": ["email", "update", "change", "account"],
     "answer": "Go to Settings > Profile > Email and enter your new email address."},

    {"question": "What is your refund policy?",
     "keywords": ["refund", "money", "return", "cancel"],
     "answer": "Refunds are processed within 5-7 business days to your original payment method."},

    {"question": "How long does shipping take?",
     "keywords": ["shipping", "delivery", "package", "time", "arrive"],
     "answer": "Standard shipping takes 3-5 business days; express takes 1-2 business days."},

    {"question": "How do I contact customer support?",
     "keywords": ["contact", "support", "help", "reach"],
     "answer": "You can reach us via live chat, email at support@example.com, or phone."},
]

STOPWORDS = {
    "i", "my", "do", "does", "is", "am", "are", "the", "a", "an",
    "how", "what", "can", "get", "to", "of", "in", "on", "for",
    "your", "you", "it", "and", "or", "back"
}

def search_by_keyword(faqs, query, top_k=3):
    """Keyword search: matches only on meaningful (non-stopword) words."""
    query_words = set(query.lower().split()) - STOPWORDS
    results = []
    for faq in faqs:
        faq_words = (set(faq["question"].lower().replace("?", "").split())
                     | set(w.lower() for w in faq["keywords"])) - STOPWORDS
        if query_words & faq_words:
            results.append((faq, 0.5))
    return results[:top_k]


In [ ]:

# FAQMatcher class

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class FAQMatcher:
    def __init__(self, faqs):
        self.faqs = faqs
        # Combine question + keywords into one corpus per FAQ
        self.corpus = [f"{faq['question']} {' '.join(faq['keywords'])}" for faq in faqs]
        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(self.corpus)

    def match(self, query, top_k=3):
        """Vectorize query, compute cosine similarity, return top_k (faq_dict, score) tuples."""
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.tfidf_matrix).flatten()

        # Pair each FAQ with its score, sort descending
        results = list(zip(self.faqs, scores))
        results.sort(key=lambda x: x[1], reverse=True)

        # Round scores to 4 decimal places, keep only top_k
        results = [(faq, round(float(score), 4)) for faq, score in results[:top_k]]
        return results

    def best_match(self, query, threshold=0.15):
        """Return the single best match if score >= threshold, else None."""
        results = self.match(query, top_k=1)
        if results and results[0][1] >= threshold:
            return results[0]
        return None

    def explain_match(self, query):
        """Return a formatted string showing top 3 matches with their scores."""
        results = self.match(query, top_k=3)
        if not results:
            return "No matches found."
        lines = [f"Top matches for: '{query}'"]
        for i, (faq, score) in enumerate(results, 1):
            lines.append(f"  {i}. [{score}] {faq['question']}")
        return "\n".join(lines)

In [ ]:

# Hybrid Search

def hybrid_search(faqs, query, top_k=3):
    matcher = FAQMatcher(faqs)

    # 1. Keyword search - base score 0.5
    keyword_results = search_by_keyword(faqs, query, top_k=len(faqs))
    keyword_scores = {faq["question"]: score for faq, score in keyword_results}

    # 2. TF-IDF matching - cosine similarity scores
    tfidf_results = matcher.match(query, top_k=len(faqs))
    tfidf_scores = {faq["question"]: score for faq, score in tfidf_results}

    # 3. Merge, keeping the HIGHEST score per FAQ
    combined = {}
    faq_lookup = {faq["question"]: faq for faq in faqs}

    for question, score in keyword_scores.items():
        combined[question] = max(combined.get(question, 0), score)

    for question, score in tfidf_scores.items():
        combined[question] = max(combined.get(question, 0), score)

    # Sort by score descending, return top_k as (faq_dict, score) tuples
    sorted_results = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    final_results = [(faq_lookup[q], round(score, 4)) for q, score in sorted_results[:top_k]]
    return final_results

In [ ]:

# Comparison Demonstration

matcher = FAQMatcher(faqs)
test_queries = [
    "I forgot my login credentials",
    "Can I get my money back?",
    "package delivery time"
]

for query in test_queries:
    print(f"Query: {query}\n")

    # Keyword Search
    print("[Keyword Search]")
    kw_results = search_by_keyword(faqs, query)
    if kw_results:
        for i, (faq, score) in enumerate(kw_results, 1):
            print(f"  {i}. [{score}] {faq['question']}")
    else:
        print("  (no results)")

    # TF-IDF Matching
    print("\n[TF-IDF Matching]")
    tfidf_results = matcher.match(query)
    for i, (faq, score) in enumerate(tfidf_results, 1):
        print(f"  {i}. [{score}] {faq['question']}")

    # Hybrid Search
    print("\n[Hybrid Search]")
    hybrid_results = hybrid_search(faqs, query)
    for i, (faq, score) in enumerate(hybrid_results, 1):
        print(f"  {i}. [{score}] {faq['question']}")

    # Best match
    best = matcher.best_match(query)
    if best:
        print(f"\nBest match: {best[0]['question']} (confidence: {best[1]})")
    else:
        print("\nBest match: None (below threshold)")

    print("\n" + "="*60 + "\n")